In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.cm as cm
import seaborn as sns
import scipy.stats as ss
import pickle
import os
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import distance
from sklearn.neighbors import KernelDensity
from scipy.stats import gaussian_kde
from rdkit.Chem import Draw
from collections import Counter
from matplotlib.colors import to_hex
from sklearn.preprocessing import RobustScaler, StandardScaler
from collections import defaultdict
from pycirclize import Circos
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import pymol
from pymol import cmd
import tarfile
from matplotlib.colors import to_rgb

In [2]:
def zoom_to_fixed_box(selection="structure", box_size=30.0, box_name="_zoom_box"):
    (xmin, ymin, zmin), (xmax, ymax, zmax) = cmd.get_extent(selection)
    cx = 0.5 * (xmin + xmax)
    cy = 0.5 * (ymin + ymax)
    cz = 0.5 * (zmin + zmax)

    h = box_size / 2.0

    cmd.delete(box_name)

    corners = [
        (cx-h, cy-h, cz-h),
        (cx-h, cy-h, cz+h),
        (cx-h, cy+h, cz-h),
        (cx-h, cy+h, cz+h),
        (cx+h, cy-h, cz-h),
        (cx+h, cy-h, cz+h),
        (cx+h, cy+h, cz-h),
        (cx+h, cy+h, cz+h),
    ]

    for i, (x, y, z) in enumerate(corners):
        cmd.pseudoatom(f"{box_name}_{i}", pos=[x, y, z])

    cmd.group(box_name, f"{box_name}_*")
    cmd.zoom(box_name, buffer=0.0, complete=1)

    # cleanup
    cmd.delete(box_name)
    cmd.delete(f"{box_name}_*")


In [4]:
# Define some paths
root = '../../../github/mtb-targeted-protein-degradation/notebooks'

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "output", "pocket_detection_data.csv"))
pocket_detection_data_interpro = pd.read_csv(os.path.join(root, "..", "output", "pocket_detection_data_interpro.tsv"), sep='\t')
pocket_to_prob = {f"{i.replace('.pdb', '')}_pocket_{j}": k for i,j,k in 
                  pocket_detection_data[['File name', 'Pocket number', 'Pocket probability']].values}
PROTEINS = sorted(set(pocket_detection_data['Uniprot AC']))
# Pocket to Interpro domains
pocket_to_domain = defaultdict(list)
for st, pocket_number, interpro in pocket_detection_data_interpro[['File name', 'Pocket number', 'Interpro curated annotation']].values:
    pocket_to_domain[f"{st.replace('.pdb', '')}_pocket_{pocket_number}"].append(interpro)

# Uniprot to gene name
uniprot_to_gene = pd.read_csv(os.path.join(root, "..", "data", "mtb_trna_synthetases_bosch_2021_fig5.csv"))
uniprot_to_gene = {i: j for i,j in zip(uniprot_to_gene['uniprot_ac'], uniprot_to_gene['gene_name_in_bosch_2021'])}

uniprot_ids = ["P9WFW5","P9WFW7","P9WFW3","P9WQA1","P9WN61","P9WFT5","P9WFV3","P9WFS9","P9WFV1","P9WFV9","P9WFT9","P9WFV7","P9WFT7",
            "P9WFW1","P9WFU5","P9WFU9","P9WFV5","P9WFT3","P9WFU3","P9WFU1","P9WFT1"]

palette = list(plt.get_cmap("tab20").colors) + [plt.get_cmap("tab20b").colors[0]]
cmap_dict = {uniprot_to_gene[uid]: to_hex(palette[i]) for i, uid in enumerate(uniprot_ids)}

# Prepare dicts
DOCKING_RESULTS = {}
DOCKING_RESULTS['Enamine-DL-HLL-100'] = dict()
DOCKING_RESULTS['EnamineREAL10B'] = dict()

# Get SMILES
ID_TO_SMILES = {}
ID_TO_SMILES['Enamine-DL-HLL-100'] = pickle.load(open(os.path.join(root, "..", "output", "enamine_characterization", "ID_TO_SMI.pkl"), "rb"))
df = pd.read_csv(os.path.join(root, "..", "output", "unidock_REAL_docking", 'inference_10B', 'clustered_compounds.csv'))
ID_TO_SMILES['EnamineREAL10B'] = {i: j for i,j in zip(df['id'], df['smiles'])}
del df

# For each pocket - ORIGINAL
PATH_TO_DOCKING_RESULTS = os.path.join(root, "..", "output", "unidock_docking", 'docking_results')
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS))):
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS, pocket, 'report.csv'))
    assert len(scores) == 100154
    DOCKING_RESULTS['Enamine-DL-HLL-100'][pocket] = {i: j for i, j in zip(scores['compound'], scores['score'])}

# For each pocket - REAL 2
PATH_TO_DOCKING_RESULTS = os.path.join(root, "..", "output", "unidock_REAL_docking_2", 'docking_results')
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS))):
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS, pocket, 'report.csv'))
    assert len(scores) == 99105
    DOCKING_RESULTS['EnamineREAL10B'][pocket] = {i: j for i, j in zip(scores['compound'], scores['score'])}

# Define pockets
POCKETS = sorted(DOCKING_RESULTS['Enamine-DL-HLL-100'])

# Define path to output
PATH_TO_OUTPUT = os.path.join(root, "..", "output", "pocket_visualisation")
os.makedirs(PATH_TO_OUTPUT, exist_ok=True)

100%|██████████| 276/276 [00:16<00:00, 16.79it/s]


In [5]:
uniprot_to_gene

{'P9WFW5': 'argS',
 'P9WFW7': 'alaS',
 'P9WFW3': 'aspS',
 'P9WQA1': 'gatA',
 'P9WN61': 'gatB',
 'P9WFT5': 'thrS',
 'P9WFV3': 'ileS',
 'P9WFS9': 'valS',
 'P9WFV1': 'leuS',
 'P9WFV9': 'gltS',
 'P9WFT9': 'proS',
 'P9WFV7': 'glyS',
 'P9WFT7': 'serS',
 'P9WFW1': 'cysS1',
 'P9WFU5': 'metS',
 'P9WFU9': 'lysS',
 'P9WFV5': 'hisS',
 'P9WFT3': 'trpS',
 'P9WFU3': 'pheS',
 'P9WFU1': 'pheT',
 'P9WFT1': 'tyrS'}

In [32]:
protein = 'P9WFU1'

# Get pockets to use and ref st
pockets_to_use = sorted([[i, pocket_to_prob[i]] for i in pocket_to_prob if protein in i], key = lambda x: x[1], reverse=True)
ref_st = pockets_to_use[0][0].split("_pocket_")[0] + ".pdb"
print(pockets_to_use)

# Load deliverables
df1 = pd.read_csv(os.path.join(root, "..", "output", "unidock_docking", "deliverable", "compounds_Enamine-DL-HLL-100.csv"))
df2 = pd.read_csv(os.path.join(root, "..", "output", "unidock_REAL_docking_2", "deliverable", "compounds_EnamineREAL10B.csv"))

# Identify compounds
gene = uniprot_to_gene[protein]
cpds1 = sorted(set(df1[df1[gene] < -10.5].sort_values(gene)['compound_id']))
cpds2 = sorted(set(df2[df2[gene] < -10.5].sort_values(gene)['compound_id']))

print(cpds1[0], cpds2[0])

[['alphafold2_P9WFU1_model_0_pocket_1', 0.726], ['swissmodel_P9WFU1_model_0_pocket_1', 0.699], ['alphafold3_P9WFU1_model_4_pocket_2', 0.538], ['alphafold3_P9WFU1_model_1_pocket_2', 0.537], ['swissmodel_P9WFU1_model_0_pocket_2', 0.514], ['chai1_P9WFU1_model_0_pocket_2', 0.453], ['alphafold2_P9WFU1_model_0_pocket_2', 0.446], ['alphafold3_P9WFU1_model_0_pocket_2', 0.442], ['alphafold3_P9WFU1_model_2_pocket_2', 0.415], ['chai1_P9WFU1_model_3_pocket_3', 0.303], ['alphafold3_P9WFU1_model_4_pocket_3', 0.274], ['alphafold3_P9WFU1_model_0_pocket_3', 0.224], ['swissmodel_P9WFU1_model_0_pocket_3', 0.208]]
Z1744517862 s_11____27039086____13030284


In [34]:
sorted(DOCKING_RESULTS['EnamineREAL10B']["alphafold2_P9WFU1_model_0_pocket_1"].values())[:5]

[-11.119, -10.907, -10.589, -10.511, -10.496]

In [35]:
pocket_to_domain['alphafold2_P9WFU1_model_0_pocket_1']

['tRNA Binding Domain', 'Other too broad/unspecified functional entities']

In [38]:
# Create pymol session
pymol.finish_launching(['pymol','-cq'])
cmd.reinitialize()

# Load structure
cmd.load(os.path.join(root, "..", "output", "aligned_relaxed_structures", protein, ref_st), "structure")
cmd.set_color("structure_color", [0.7804, 0.8275, 0.8667])
cmd.color("structure_color", "structure")
cmd.show("surface", "structure")
cmd.hide("cartoon", "structure")
cmd.hide("lines", "structure")
cmd.hide("sticks", "structure")
cmd.set("transparency", 0.3, "structure")

# Load pockets
path_to_pockets = os.path.join(root, "..", "output", "detected_pockets")
for pocket in pockets_to_use:
    pocket, prob = pocket
    st = pocket.split("_pocket_")[0]
    pocket_number = pocket.split("_pocket_")[1]
    path_to_pocket = os.path.join(path_to_pockets, protein, st, 'pockets', f"pocket_{pocket_number}.pdb")
    cmd.load(path_to_pocket, pocket)

    pocket_color = list(to_rgb(cmap_dict[uniprot_to_gene[protein]]))
    cmd.set_color(f"{pocket}_color", pocket_color)
    cmd.color(f"{pocket}_color", pocket)

    ligands_loaded = []

    for cpd1 in tqdm(cpds1[:1]):
        p = os.path.join(root, "..", "output", "unidock_docking", 'docking_results', pocket)
        with tarfile.open(os.path.join(p, "docking.tar.gz"), "r|gz") as tf:
            data = None
            for member in tf:
                if member.name == f"docking/{cpd1}_out.sdf":
                    data = tf.extractfile(member).read()
                    break
        tmp_sdf = os.path.join(p, f"{cpd1}_tmp.sdf")
        with open(tmp_sdf, "w") as f:
            f.write(data.decode("utf-8", errors="replace"))
        lig = f"{cpd1}"
        cmd.load(tmp_sdf, lig)
        os.remove(tmp_sdf)
        ligands_loaded.append(lig)
        cmd.util.cbag(lig)
        cmd.set_color("ligC_orange", [0xF5/255, 0xA6/255, 0x3A/255])
        cmd.color("ligC_orange", f"{lig} and elem C")
    for cpd1 in tqdm(cpds2[:1]):
        p = os.path.join(root, "..", "output", "unidock_REAL_docking_2", 'docking_results', pocket)
        with tarfile.open(os.path.join(p, "docking.tar.gz"), "r|gz") as tf:
            data = None
            for member in tf:
                if member.name == f"docking/{cpd1}_out.sdf":
                    data = tf.extractfile(member).read()
                    break
        tmp_sdf = os.path.join(p, f"{cpd1}_tmp.sdf")
        with open(tmp_sdf, "w") as f:
            f.write(data.decode("utf-8", errors="replace"))
        lig = f"{cpd1}"
        cmd.load(tmp_sdf, lig)
        os.remove(tmp_sdf)
        ligands_loaded.append(lig)
        cmd.util.cbag(lig)
        cmd.set_color("ligC_orange", [0xF5/255, 0xA6/255, 0x3A/255])
        cmd.color("ligC_orange", f"{lig} and elem C")

    # Color all residues within 4 A of any ligand atom
    if ligands_loaded:
        lig_sel = " or ".join(ligands_loaded)
        near_sel = f"{pocket}_near_lig_res"
        cmd.select(near_sel, f"byres (structure within 4 of ({lig_sel}))")
        cmd.set_color("near_lig_blue", pocket_color)
        cmd.color("near_lig_blue", near_sel)

    cmd.delete(pocket)

    break

# Fancy visualization
cmd.bg_color("white")
cmd.set("orthoscopic", 1)
cmd.set("depth_cue", 0)
cmd.set("ray_trace_fog", 0)
cmd.set("ray_shadows", 0)
cmd.set("ray_trace_mode", 1)
cmd.set("ray_trace_gain", 0.02)
cmd.set("spec_reflect", 0)
cmd.set("specular", 0)
cmd.set("antialias", 2)
zoom_to_fixed_box("structure", box_size=100)
cmd.save(os.path.join(PATH_TO_OUTPUT, f"session_{protein}_.pse"))
# cmd.ray(1200, 1200)
# cmd.png(os.path.join(PATH_TO_OUTPUT, f"figure_{protein}.png"), dpi=600)

100%|██████████| 1/1 [00:01<00:00,  1.87s/it]
